# 2. Inference on Prediction Data

In [ ]:
# # # # # # # # # # # #
# Prediction with Autogluon

import pandas as pd
import numpy as np
import joblib
from autogluon.tabular import TabularPredictor

# Prediction data
df = pd.read_csv("predict.csv")
claim_ids = df['claim_number']

# Claim_date and Weekends
df['claim_date'] = pd.to_datetime(df['claim_date'], errors='coerce')
df['is_weekend_claim'] = df['claim_date'].dt.dayofweek.isin([5, 6]).astype(int)

# Seasons
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    return 'Fall'
df['season'] = df['claim_date'].dt.month.apply(get_season)

# Encoder (training)
from sklearn.preprocessing import LabelEncoder
le_season = LabelEncoder()
df['season'] = le_season.fit_transform(df['season'])
joblib.dump(le_season, "season_label_encoder.pkl")

# Liability features
df['liab_prct'] = pd.to_numeric(df['liab_prct'], errors='coerce').fillna(0)
df['liability_minus_risk'] = df['liab_prct'] - df['safety_rating']

df['provability_score'] = (
    (df['witness_present_ind'] == 'Y').astype(int).fillna(0)
    + df['policy_report_filed_ind'].fillna(0)
    + df['email_or_tel_available'].fillna(0)
    + (df['in_network_bodyshop'] == 'yes').astype(int).fillna(0))

df['claim_est_payout'] = df['claim_est_payout'].fillna(0)

# Liability Binning
bins = list(range(20, 60, 4))
labels = [f"{b}-{b+4}" for b in bins[:-1]]
df['liab_bin'] = pd.cut(df['liab_prct'], bins=bins, labels=labels, include_lowest=True)
df['liab_bin'] = df['liab_bin'].astype(str).fillna("Other")

# Above Liability Threshold
df['above_liability_threshold'] = (df['liab_prct'] > 50).astype(int)

# Log Transformations
df['annual_income'] = np.log1p(df['annual_income'])

df['claim_est_payout'] = np.log1p(df['claim_est_payout'])

# Claims binning
df['claims_binned'] = pd.cut(
    df['past_num_of_claims'],
    bins=[-1, 0, 3, float('inf')],
    labels=['None', 'Medium', 'High']).astype('category')

# Data preprocessing
df['witness_flag'] = (df['witness_present_ind'] == 'Y').astype(int)

df['accident_type'] = df['accident_type'].astype(str)
df['accident_site'] = df['accident_site'].astype(str)

# liability x accident type
for t in df['accident_type'].unique():
    safe_t = t.replace('/', '_')
    col = f"liab_if_type_{safe_t}"
    df[col] = df['liab_prct'] * (df['accident_type'] == t).astype(int)

# liability x accident site
for s in df['accident_site'].unique():
    safe_s = s.replace('/', '_')
    col = f"liab_if_site_{safe_s}"
    df[col] = df['liab_prct'] * (df['accident_site'] == s).astype(int)

# liability x witness
df['liab_if_witness']    = df['liab_prct'] * df['witness_flag']
df['liab_if_nowitness']  = df['liab_prct'] * (1 - df['witness_flag'])

# witness x accident type
for t in df['accident_type'].unique():
    safe_t = t.replace('/', '_')
    col = f"witness_if_type_{safe_t}"
    df[col] = df['witness_flag'] * (df['accident_type'] == t).astype(int)

# witness x accident site
for s in df['accident_site'].unique():
    safe_s = s.replace('/', '_')
    col = f"witness_if_site_{safe_s}"
    df[col] = df['witness_flag'] * (df['accident_site'] == s).astype(int)

# accident type x accident site
for t in df['accident_type'].unique():
    for s in df['accident_site'].unique():
        safe_t = t.replace('/', '_')
        safe_s = s.replace('/', '_')
        col = f"type_{safe_t}__site_{safe_s}"
        df[col] = ((df['accident_type'] == t).astype(int) *
                   (df['accident_site'] == s).astype(int))

# liability x witness x accident type x accident site
df['liab_type_site_witness'] = (
    df['liab_prct'] *
    df['witness_flag'] *
    df['accident_type'].astype('category').cat.codes *
    df['accident_site'].astype('category').cat.codes
)

# nonlinear features
df['liab_prct_sq'] = df['liab_prct'] ** 2
df['liab_prct_cu'] = df['liab_prct'] ** 3

# recoverable amounts
df['recoverable_amount'] = ((100 - df['liab_prct']) / 100) * df['claim_est_payout']
df['recoverable_amount_log'] = np.log1p(df['recoverable_amount'])

# liability distance from threshold at 50
df['liab_dist_50'] = df['liab_prct'] - 50
df['liab_dist_abs_50'] = df['liab_dist_50'].abs()

# using training means (for testing / prediction)
if 'subrogation' not in df.columns:
    means = joblib.load("liab_means.pkl")
    mean_liab_sub = means['sub']
    mean_liab_non = means['non']

# submeans
df['liab_dist_submean'] = df['liab_prct'] - mean_liab_sub
df['liab_dist_nonmean'] = df['liab_prct'] - mean_liab_non

# claims velocity
df['claim_velocity'] = (
        (df['past_num_of_claims'] == 0) * 0.0 +
        (df['past_num_of_claims'].between(1, 3)) * 0.5 +
        (df['past_num_of_claims'] >= 4) * 1.0
)

df['claim_velocity_scaled'] = df['claim_velocity'] * df['liab_prct'] / 100.0

# remove columns
df = df.drop(columns=['claim_number', 'zip_code', 'recoverable_amount'],
             errors='ignore')

In [ ]:
# Load previously trained model
path = "PATH/TO/MODEL"
predictor = TabularPredictor.load(path)
preds = predictor.predict(df)

# Export results
submission = pd.DataFrame({
    "claim_number": claim_ids,
    "subrogation": preds
})

submission.to_csv("predictions.csv", index=False)
print("Predictions saved to predictions.csv")
